# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadbinijaz17/flyrankAI_Intern_ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Bootstrap: run identically in Colab and locally (copied from the starter notebooks).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muhammadbinijaz17/flyrankAI_Intern_ML"
REPO_DIR = "flyrankAI_Intern_ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # walk up to the repo root (folder containing data/raw)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Working dir:", os.getcwd())
print("Starter data found. You're ready.")

Working dir: D:\FlyrankAI\flyrankAI_Intern_ML
Starter data found. You're ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Unit of Analysis (Grain):** **One row = one pseudonymized content item (page)**, uniquely identified by `content_id` (a hash starting with `content_` + 12 hex characters), belonging to one of 32 pseudonymized `client_id`s. There are exactly 30,000 distinct content items.
- **Time Window:** A **trailing 90-day aggregate snapshot** ending at dataset export time.
  - Every content item in this slice has an age of at least 90 days (`content_age_days >= 90`), ensuring equal 90-day exposure time.
  - The 90-day window is decomposed into two 30-day comparison sub-windows:
    - **Previous 30-day window (`*_prev_30d`):** Days 31 to 60 back from export (the baseline period).
    - **Last 30-day window (`*_last_30d`):** Days 1 to 30 back from export (the recent outcome period).
  - Activity totals (`impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, etc.) span the entire 90-day window.
- **Output of Analysis:** A prioritized review queue assigning each content item a predicted probability / priority score of decline, enabling human SEO editors to inspect the top-ranked pages first.

In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Verify Grain (Unit of Analysis)
total_rows = len(df)
unique_content_ids = df["content_id"].nunique()
duplicate_content_ids = df.duplicated(subset=["content_id"]).sum()
unique_clients = df["client_id"].nunique()

print(f"Total rows in dataset: {total_rows:,}")
print(f"Unique content_id count: {unique_content_ids:,}")
print(f"Duplicate content_ids: {duplicate_content_ids}")
print(f"Unique client_id count: {unique_clients}")
assert unique_content_ids == total_rows == 30_000, "Grain violation: content_id is not unique per row!"
assert duplicate_content_ids == 0, "Duplicate rows detected!"

# 2. Verify Time Window Constraints
min_age = df["content_age_days"].min()
max_age = df["content_age_days"].max()
max_impressions_days = df["days_with_impressions"].max()
max_sessions_days = df["days_with_sessions"].max()

print(f"\nContent age range: [{min_age}, {max_age}] days (all >= 90 days)")
print(f"Max days with impressions in window: {max_impressions_days} (bounded by 90)")
print(f"Max days with sessions in window: {max_sessions_days} (bounded by 90)")
assert min_age >= 90, "Found items younger than the 90-day window!"

Total rows in dataset: 30,000
Unique content_id count: 30,000
Duplicate content_ids: 0
Unique client_id count: 32

Content age range: [90, 564] days (all >= 90 days)
Max days with impressions in window: 88 (bounded by 90)
Max days with sessions in window: 90 (bounded by 90)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every one of the 44 raw columns in `content_refresh_anonymized.csv` (plus the derived target) is assigned to exactly one category:

### 1. Features (Safe Signals Knowable Prior to Prediction)
- **Search Performance (GSC):** `impressions_90d`, `clicks_90d`, `ctr` (rate $\times 100$), `avg_position` ($0 = \text{no data}$), `days_with_impressions`, `impressions_prev_30d`, `clicks_prev_30d`.
- **User Engagement (GA4):** `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `engagement_rate`, `scroll_events_90d`, `scroll_rate` (can exceed 100%), `ai_sessions_90d`, `ai_traffic_pct` (can exceed 100%), `days_with_sessions`, `sessions_prev_30d`.
- **Content & Keyword Attributes:** `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`.
- **Discretized Tiers:** `age_tier`, `age_tier_order`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`.

### 2. Label / Proxy (The Target Being Predicted)
- **`is_declining_label`:** Binary indicator (1 if `trend_direction == "down"`, 0 otherwise). 16,262 rows (54.2% base rate).
- *Label construction inputs (NEVER features):* `trend_direction` and `trend_pct`.

### 3. Context (Grouping, Splitting & Joins Only — Never Model Features)
- **`content_id`:** Unique page identifier (grain key).
- **`client_id`:** 32 pseudonymized accounts. Crucial for group-aware splitting (`GroupKFold` / client-holdout) so models are evaluated on unseen client domains.

### 4. Excluded Fields (With Explicit Rationales)
- **`trend_direction` & `trend_pct`:** **Label Leakage.** `is_declining_label` is computed directly from `trend_direction == "down"`, which is defined as `trend_pct < -20.0`. Allowing either into a model produces a tautological 1.0 AUC that learns zero real signal.
- **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`:** **Outcome Window Leakage.** These metrics represent performance during the target outcome window ($T_{\text{last30}}$) used to determine decline. In production forecasting, last-30-day outcomes are not known at prediction time.
- **`provider_used` & `model_used`:** **Irrelevant Tooling Metadata.** Reflects backend LLM generator choice (`openai`, `google`, `gemini-2.5-flash`), with high sparsity / missingness. Not intrinsic signals of content search intent, search demand, or user behavior.

In [5]:
# Section 2 check: Exhaustive classification of all 44 columns + target
features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "search_volume", "competition",
    "competition_level", "cpc", "content_type", "main_intent", "word_count",
    "char_count", "content_age_days", "days_since_last_update", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "age_tier", "age_tier_order", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier"
]

labels_and_sources = ["trend_direction", "trend_pct"]
context = ["content_id", "client_id"]
excluded = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "provider_used", "model_used"
]

all_columns = set(df.columns)
classified_columns = set(features + labels_and_sources + context + excluded)

print(f"Total raw columns: {len(all_columns)}")
print(f"Classified columns: {len(classified_columns)}")
print(f"Features: {len(features)}")
print(f"Label & Sources: {len(labels_and_sources)}")
print(f"Context: {len(context)}")
print(f"Excluded: {len(excluded)}")

assert classified_columns == all_columns, f"Unclassified columns: {all_columns - classified_columns}"

# Verify Target Leakage Tautology
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"\nLabel distribution (is_declining_label == 1): {df['is_declining_label'].mean():.2%}")
print(f"Correlation between is_declining_label and trend_direction=='down': {(df['is_declining_label'] == (df['trend_direction'] == 'down')).mean():.1%}")

Total raw columns: 44
Classified columns: 44
Features: 35
Label & Sources: 2
Context: 2
Excluded: 5

Label distribution (is_declining_label == 1): 54.21%
Correlation between is_declining_label and trend_direction=='down': 100.0%


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below we verify every contract assertion with explicit data checks:
1. **Grain Assertion:** Exactly 0 content items appear more than once.
2. **Class & Client Balance:** 32 clients; positive label rate is 54.21% (16,262 / 30,000).
3. **Missingness Structure (Systematic vs Random):**
   - `search_volume`, `competition`, `cpc`: Missing for 2,468 rows. Grouped by `content_type`: exactly 100% of `feedly article` rows have no keyword data, whereas `comparison article` has 0% missing and `keyword article` has 1.37% missing.
   - `word_count`, `char_count`: Missing for 7,699 rows. All 7,699 missing rows are in `keyword article` (28.3% missing rate within that type); 0% missing in other types.
   - `trend_pct`: Missing for 3,388 rows where `impressions_prev_30d == 0` (division by zero).
4. **Special Encodings & Value Boundaries:**
   - `avg_position == 0`: 1,205 rows represent "no position data recorded", not rank #0.
   - Rates $\times 100$: `ctr` mean is 1.62% (0.0162 in raw percentage), max is 100.0%.
   - `scroll_rate` and `ai_traffic_pct` exceed 100% due to independent multi-event tracking and independent attribution systems.

In [7]:
# 1. Grain Query Check
grain_check = df.groupby("content_id").size().reset_index(name="counts")
violations = grain_check[grain_check["counts"] > 1]
print(f"1. Grain violations (rows with count > 1): {len(violations)}")
assert len(violations) == 0, "Grain query failed: duplicate content_ids exist!"

# 2. Counts and Label Base Rate
print(f"\n2. Total dataset size: {len(df):,} rows across {df['client_id'].nunique()} clients.")
label_counts = df["trend_direction"].value_counts()
print("Trend direction breakdown:\n", label_counts)
print(f"Positive label ('down') count: {(df['trend_direction'] == 'down').sum():,} ({df['is_declining_label'].mean():.2%})")

# 3. Missingness & Systematic Pattern by content_type
print("\n3. Missing value rates per column:")
missing_summary = df.isna().mean().sort_values(ascending=False)
print(missing_summary[missing_summary > 0].apply(lambda x: f"{x:.2%}"))

print("\nSystematic Missingness by content_type:")
missing_by_type = df.groupby("content_type")[["search_volume", "word_count", "cpc"]].apply(lambda g: g.isna().mean())
print(missing_by_type.map(lambda x: f"{x:.1%}"))

# 4. Zero-Position and Bound Checks
zero_pos_count = (df["avg_position"] == 0).sum()
print(f"\n4. avg_position == 0 ('no data' indicator): {zero_pos_count:,} rows ({(zero_pos_count/len(df)):.2%})")
print(f"scroll_rate > 100% count: {(df['scroll_rate'] > 100).sum():,} rows (max: {df['scroll_rate'].max():.1f}%)")
print(f"ai_traffic_pct > 100% count: {(df['ai_traffic_pct'] > 100).sum():,} rows (max: {df['ai_traffic_pct'].max():.1f}%)")

1. Grain violations (rows with count > 1): 0

2. Total dataset size: 30,000 rows across 32 clients.
Trend direction breakdown:
 trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
Positive label ('down') count: 16,262 (54.21%)

3. Missing value rates per column:
provider_used        71.46%
word_count           25.66%
char_count           25.66%
char_count_tier      25.66%
word_count_tier      25.66%
model_used           19.11%
trend_pct            11.29%
competition_level     8.70%
cpc                   8.23%
search_volume         8.23%
competition           8.23%
main_intent           7.91%
scroll_rate           0.42%
dtype: str

Systematic Missingness by content_type:
                   search_volume word_count     cpc
content_type                                       
comparison article          0.0%       0.0%    0.0%
feedly article            100.0%       0.0%  100.0%
keyword article             1.4%      28.3%

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

An honest data contract requires explicit boundaries on what inferences the data can and cannot support:

1. **Contemporaneous Window (Static Slice vs Longitudinal Panel):**
   - In this 30,000-row slice, features and label are aggregated from the *same* trailing 90-day window. This setup tests whether current signals correlate with recent decline ("is this page declining now?"), but it does not represent a true prospective forward forecast (e.g. predicting month $M+1$ strictly from month $M$ and earlier).
2. **Systematic Missingness vs Zero Performance:**
   - Missing `search_volume` (100% of feedly articles) or missing `word_count` (28.3% of keyword articles) is an artifact of ingestion pipelines, NOT evidence of 0 search volume or a 0-word article. Imputing with blind zeros injects artificial category signals; missingness must be tracked with explicit boolean flags (`has_search_volume`, `has_word_count`).
3. **Measurement System Divergence (GA4 vs GSC vs AI Referrals):**
   - `scroll_rate` and `ai_traffic_pct` can exceed 100% because numerators (scroll events, AI sessions) and denominators (pageviews, total sessions) are captured by independent client-side scripts.
   - `ai_sessions_90d` measures *referral click-through traffic* from generative AI engines (ChatGPT, Claude, Perplexity), NOT whether the page was cited in Google AI Overviews or ranked in AI Search.
4. **Warehouse Panel Transitions (Looking Ahead to Weeks 3+):**
   - The full warehouse panel (~79M rows) contains unbalanced client histories (`gsc_data_start`, `ga4_data_start`). Early dates for some clients have GSC search data but zero-filled GA4 engagement with `ga4_data_available = FALSE` or NULL. Any warehouse model must filter on access availability flags rather than mistaking uninstrumented periods for zero engagement.

In [9]:
# Section 4 check: Demonstrate data boundaries and anomalies

# 1. Inspect rate anomalies (> 100%)
anomalous_rates = df[df["scroll_rate"] > 100][["content_id", "content_type", "pageviews_90d", "scroll_events_90d", "scroll_rate"]].head(3)
print("1. Examples of scroll_rate > 100% (multiple scroll events per pageview):")
print(anomalous_rates.to_string(index=False))

# 2. Inspect avg_position == 0 distribution
no_pos_df = df[df["avg_position"] == 0]
print(f"\n2. avg_position == 0 pages: Median impressions = {no_pos_df['impressions_90d'].median()}, Median clicks = {no_pos_df['clicks_90d'].median()}")
print("   (Confirms avg_position=0 denotes unranked/low-signal pages, not position #0)")

# 3. Missing prev_30d volume resulting in undefined trend_pct
zero_prev_impr = df[df["impressions_prev_30d"] == 0]
print(f"\n3. Pages with impressions_prev_30d == 0: {len(zero_prev_impr):,} rows")
print(f"   Corresponding trend_direction breakdown:\n{zero_prev_impr['trend_direction'].value_counts().to_dict()}")

1. Examples of scroll_rate > 100% (multiple scroll events per pageview):
          content_id    content_type  pageviews_90d  scroll_events_90d  scroll_rate
content_d8a23b5e10c5 keyword article              1                  2        200.0
content_5d77d3077984 keyword article              1                  3        300.0
content_4ff3d119662e keyword article              1                  2        200.0

2. avg_position == 0 pages: Median impressions = 1.0, Median clicks = 0.0
   (Confirms avg_position=0 denotes unranked/low-signal pages, not position #0)

3. Pages with impressions_prev_30d == 0: 3,388 rows
   Corresponding trend_direction breakdown:
{'new': 2236, 'flat': 1152}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.